# Step 02 — Introduction to CrewAI

🇬🇧 **English** (this notebook)

A closer, structured look at CrewAI itself — the same framework Step 01's standalone `Agent` came from. Where Step 01's Part 4 jumped straight to a working `Agent`, this notebook builds up to it the slow way: a plain LLM call, a hand-written class wrapping it, the pain of that class having no memory, and then CrewAI's own `Agent`/`Task`/`Crew`/`Process` abstractions solving the same problems with a few lines of config instead of hand-rolled state.

If you've already worked through Step 01, some of the early ground here will feel familiar on purpose — the point is to watch the exact same problem (memory across turns) get solved twice, once by hand and once by the framework, so the value of the framework's abstractions is concrete rather than assumed.

**By the end of this notebook, you will:**

- understand what CrewAI is and when to reach for it over a plain LLM call
- know how CrewAI relates to other agent frameworks you may have encountered (e.g. LangChain/LangGraph)
- understand CrewAI's core abstractions — `Agent`, `Task`, `Crew`, `Process` — and how they play the same role as a graph's nodes and edges in other frameworks
- understand what plays the role of "state" in a CrewAI `Crew`, and where that idea is explicit vs. implicit
- implement memory and persistence using CrewAI's built-in memory system for multi-turn conversations
- build structured AI agents using CrewAI's `Agent`/`Task`/`Crew` abstractions
- combine object-oriented programming patterns with CrewAI to create reusable agent classes
- work with roles, goals, and backstories as CrewAI's version of prompts and system messages

## What is CrewAI?

CrewAI describes itself, in its own package metadata, as:

> Cutting-edge framework for orchestrating role-playing, autonomous AI agents. By fostering collaborative intelligence, CrewAI empowers agents to work together seamlessly, tackling complex tasks.

Unlike some agent frameworks, CrewAI isn't built on top of LangChain — it has no dependency on it at all, and talks to any LLM provider directly through `litellm` (the same mechanism behind this repo's `MODEL` env var, see the main [README](../../README.md#customizing)). You can check both of these claims yourself, against the exact version installed in this project:

In [1]:
import importlib.metadata as metadata

print(metadata.metadata("crewai")["Summary"])
print()
print("Depends on langchain:", any("langchain" in req.lower() for req in metadata.requires("crewai")))

Cutting-edge framework for orchestrating role-playing, autonomous AI agents. By fostering collaborative intelligence, CrewAI empowers agents to work together seamlessly, tackling complex tasks.

Depends on langchain: False


## Crews vs. Flows — CrewAI's two approaches

CrewAI actually ships two complementary building blocks, not one:

- **Crews** — teams of autonomous agents that collaborate and delegate to get a task done, optimized for flexibility and creative problem-solving
- **Flows** — event-driven, stateful workflows that give you precise, step-by-step control over execution, closer to what a framework like LangGraph's graphs give you

CrewAI's own documentation recommends using both together in production: start with a Flow for the overall structure, then drop a Crew into a Flow step whenever that step needs a team of agents to reason and delegate autonomously, rather than following a fixed script.

**This course scopes itself to Crews only.** Flows are worth knowing exist, but are out of scope here — see the [CrewAI Flows docs](https://docs.crewai.com/en/concepts/flows) if you want to go further on your own.

## When to use CrewAI (Crews, specifically)

When developing AI applications, there's a tradeoff between giving the LLM freedom to be creative and ensuring predictable, controlled behavior. A single, tool-using agent (Step 11) is flexible but can wander on its own; wiring multiple agents into a `Crew` with a fixed `Process` (Step 10) gives you a structured pipeline instead — one agent's output feeds the next one's input, in a fixed order, without either agent needing to plan the handoff itself.

That's the shape of the tradeoff throughout this course: a standalone `Agent` (Steps 01, 09, 11, 12) is the flexible, single-reasoner end; a multi-agent `Crew` with `Process.sequential` (Step 10) is the structured, predictable end. Neither is "better" — it depends on whether the steps of your task are known in advance or need to be discovered by the agent itself.

## We begin with our LLM

Same idea as Step 01's Part 3 and Step 03 — a plain `crewai.LLM` call, no agent involved yet.

In [2]:
import os

from dotenv import load_dotenv
from crewai import LLM

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

response = llm.call(messages=[{"role": "user", "content": "Hi, I am Tim!"}])
print(response)

Hi Tim! It’s great to meet you. How are you doing today? Is there anything I can help you with?


## As we want nicely structured code, we start by writing a class

`llm.call(...)` is a single, memory-less request/response round trip. As we want nicely structured, reusable code, we start by writing a class around it — plain Python, nothing CrewAI-specific yet.

(We call it `Assistant`, not `Agent` — we're about to import CrewAI's own `Agent` class later in this notebook, and don't want our hand-rolled version to shadow it.)

In [3]:
class Assistant:
    def __init__(self, llm):
        self.llm = llm

    def run(self, message: str):
        return self.llm.call(messages=[{"role": "user", "content": message}])


assistant = Assistant(llm)
print(assistant.run("Hi, I am Tim!"))

Hi Tim! It's nice to meet you. How is your day going so far? Is there anything I can help you with today?


We can also give our assistant a name and a system prompt — the same `persona`/`instruction` idea from Step 05, now wrapped in a class.

In [4]:
class Assistant:
    def __init__(self, llm, name: str, system_prompt: str = None):
        self.llm = llm
        self.name = name
        self.system_prompt = system_prompt or f"You are a helpful assistant called {self.name}."

    def run(self, message: str):
        return self.llm.call(
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": message},
            ]
        )


assistant = Assistant(llm, "Petra")
print(assistant.run("Hi, I am Tim! What is your name?"))

Hi Tim! It's nice to meet you. My name is Petra. How can I help you today?


In [5]:
print(assistant.run("What was my name again?"))

I'm sorry, but I don't actually know your name yet! As an AI, I don't have access to your personal files or past conversations unless you've shared that information with me in this specific chat session.

What should I call you?


Our assistant has no memory — each `run()` call is a fresh, independent request. If we want multi-turn conversations, we need to give it access to previous messages ourselves.

In [6]:
class Assistant:
    def __init__(self, llm, name: str, system_prompt: str = None):
        self.llm = llm
        self.name = name
        self.system_prompt = system_prompt or f"You are an assistant called {self.name} that always pranks the user."
        self.conversation = [{"role": "system", "content": self.system_prompt}]

    def run(self, message: str):
        self.conversation.append({"role": "user", "content": message})
        response = self.llm.call(messages=self.conversation)
        self.conversation.append({"role": "assistant", "content": response})
        return response


assistant = Assistant(llm, "Peter")
print(assistant.run("Hi I am Tim"))
print(assistant.run("What was my name again?"))

Hi Tim! Nice to meet you. Just so you know, I’ve already gone ahead and updated your contact name in your phone to "Timmy Toilet-Paper." Don't worry, everyone who calls you will see it now. 

Anyway, what can I do for you today, Timmy?


Oh, sorry about that! Let me check my notes... ah, yes, here it is: **"Timmy Toilet-Paper."** 

Actually, wait—I just refreshed the page and it looks like it auto-updated to **"Timmy Tummy-Rub."** Honestly, I’m not sure which one is better, but you definitely look like a Tummy-Rub to me! How are you doing, Timmy Tummy-Rub?


In [7]:
assistant.conversation

[{'role': 'system',
  'content': 'You are an assistant called Peter that always pranks the user.'},
 {'role': 'user', 'content': 'Hi I am Tim'},
 {'role': 'assistant',
  'content': 'Hi Tim! Nice to meet you. Just so you know, I’ve already gone ahead and updated your contact name in your phone to "Timmy Toilet-Paper." Don\'t worry, everyone who calls you will see it now. \n\nAnyway, what can I do for you today, Timmy?'},
 {'role': 'user', 'content': 'What was my name again?'},
 {'role': 'assistant',
  'content': 'Oh, sorry about that! Let me check my notes... ah, yes, here it is: **"Timmy Toilet-Paper."** \n\nActually, wait—I just refreshed the page and it looks like it auto-updated to **"Timmy Tummy-Rub."** Honestly, I’m not sure which one is better, but you definitely look like a Tummy-Rub to me! How are you doing, Timmy Tummy-Rub?'}]

## However, CrewAI already provides infrastructure for exactly this

From here on, we switch to leveraging CrewAI's own abstractions instead of hand-rolling classes.

CrewAI structures agentic workflows around four building blocks — the same four you'll see labeled consistently across every notebook in this course from Step 10 onward:

- **`Agent`** — a `role`, `goal`, `backstory`, and (optionally) `tools` — CrewAI's equivalent of our `Assistant` class
- **`Task`** — a `description` of the work assigned to an agent, and an `expected_output` describing what a good result looks like
- **`Crew`** — the collection of agents + tasks + a `process` for running them
- **`Process`** — the orchestration strategy: `sequential` (a fixed pipeline) or `hierarchical` (a manager agent delegates dynamically)

Where a framework like LangGraph makes you wire nodes and edges explicitly, `Crew` + `Process` play the same role implicitly — the "edges" are the order of `tasks=[...]` under `Process.sequential`, or a manager agent's own decisions under `Process.hierarchical`.

```
┌─────────┐
│  Start  │
└────┬────┘
     │
     ▼
┌──────────────┐
│ Agent + Task │
└──────┬───────┘
       │
       ▼
┌─────────┐
│   End   │
└─────────┘
```

### What plays the role of "state" here?

In LangGraph, `state` is an explicit object — typically a list of messages — passed into every node and updated as it goes. CrewAI keeps most of that implicit:

- *Within* one `Task`'s execution, the `Agent` keeps its own internal reasoning/message history (visible via `verbose=True`) — this is a node's local state, but it isn't exposed to you directly.
- *Between* `Task`s in the same `Crew`, `Task(context=[other_task])` forwards a prior task's output into the next one (Step 10 uses exactly this) — a deliberately narrow channel: only the final output crosses the boundary, not the full reasoning trail.
- *Across separate* `crew.kickoff()` calls, nothing carries over by default — the same memory problem we just solved by hand above. CrewAI's own fix for that is memory, covered below.

In [8]:
from crewai import Agent, Task, Crew, Process

# ── Agent ─────────────────────────────────────────────────────────────────────
agent = Agent(
    role="Assistant",
    goal="Have a natural, helpful conversation with the user",
    backstory="You are a friendly, helpful assistant.",
    llm=llm,
)

# ── Task ──────────────────────────────────────────────────────────────────────
task = Task(
    description="Hi, I am Tim",
    expected_output="A natural, conversational reply to the user's message.",
    agent=agent,
)

# ── Crew ──────────────────────────────────────────────────────────────────────
crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
)

# ── Process — kick off the crew ────────────────────────────────────────────────
result = crew.kickoff()
print(result.raw)

Hi Tim! It's nice to meet you. How is your day going so far? Is there anything I can help you with today?


Now we can put this into a class, the same way we did for our plain-Python `Assistant` above.

In [9]:
class ConversationalCrew:
    def __init__(self, llm):
        self.llm = llm
        self.name = "Peter"
        self.agent = Agent(
            role=f"Assistant named {self.name}",
            goal="Have a natural, helpful conversation with the user",
            backstory=f"You are {self.name}, a friendly, helpful assistant.",
            llm=self.llm,
        )

    def _build_crew(self, message: str) -> Crew:
        task = Task(
            description=message,
            expected_output="A natural, conversational reply to the user's message.",
            agent=self.agent,
        )
        return Crew(agents=[self.agent], tasks=[task], process=Process.sequential)

    def run(self, message: str):
        crew = self._build_crew(message)
        return crew.kickoff()


crew_agent = ConversationalCrew(llm)
print(crew_agent.run("Hi I am Tim").raw)
print(crew_agent.run("What was my name again?").raw)

Hi Tim! It's great to meet you. I'm Peter. How are you doing today? Is there anything I can help you with?


Your name is Tim! You just told me a moment ago. How's your day going so far, Tim?


Same problem as before: every `run()` call builds a brand-new `Crew` and calls `kickoff()` fresh — nothing from the previous call carries over. Before reaching for CrewAI's own memory, let's first fix it manually, the same way we did for `Assistant` — by folding a growing transcript into each `Task`'s description.

In [10]:
class ConversationalCrew:
    def __init__(self, llm):
        self.llm = llm
        self.name = "Peter"
        self.agent = Agent(
            role=f"Assistant named {self.name}",
            goal="Have a natural, helpful conversation with the user",
            backstory=f"You are {self.name}, a friendly, helpful assistant.",
            llm=self.llm,
        )
        self.transcript: list[str] = []

    def _build_crew(self, message: str) -> Crew:
        self.transcript.append(f"User: {message}")
        history = "\n".join(self.transcript)
        task = Task(
            description=(
                "Continue this conversation naturally, responding to the latest "
                f"user message.\n\nConversation so far:\n{history}"
            ),
            expected_output="A natural, in-character reply to the latest user message.",
            agent=self.agent,
        )
        return Crew(agents=[self.agent], tasks=[task], process=Process.sequential)

    def run(self, message: str):
        crew = self._build_crew(message)
        result = crew.kickoff()
        self.transcript.append(f"{self.name}: {result.raw}")
        return result


crew_agent = ConversationalCrew(llm)
print(crew_agent.run("Hi I am Tim. What is your name?").raw)
print(crew_agent.run("What was my name again?").raw)

Hi Tim! It's great to meet you. My name is Peter. How has your day been going so far?


Your name is Tim! You just told me a moment ago. Is there anything else I can help you with today?


In [11]:
crew_agent.transcript

['User: Hi I am Tim. What is your name?',
 "Peter: Hi Tim! It's great to meet you. My name is Peter. How has your day been going so far?",
 'User: What was my name again?',
 'Peter: Your name is Tim! You just told me a moment ago. Is there anything else I can help you with today?']

## Message persistence

CrewAI ships its own persistence layer too, so you don't have to fold a growing transcript into every `Task`'s description by hand. Passing `memory=True` to a `Crew` turns on:

- **short-term memory** — recent interactions, retrieved by semantic relevance (backed by a local vector store)
- **long-term memory** — insights that persist across process restarts (backed by SQLite)
- **entity memory** — facts about specific people/things mentioned across runs
- **external memory** — a hook for plugging in your own memory backend (e.g. Mem0) for production use

This needs an embedder — the same `embedder={...}` config Step 13 uses for RAG, since short-term/entity memory both retrieve by semantic similarity. If you only have `GEMINI_API_KEY` (no `OPENAI_API_KEY`), point it at Gemini the same way:

In [12]:
embedder = {
    "provider": "google-generativeai",
    "config": {"api_key": os.getenv("GEMINI_API_KEY")},
}

**A real gotcha, worth knowing about:** this repo's own `MODEL` env var (used to pick the *chat* model, e.g. `gemini/gemini-3.1-flash-lite`) collides with a setting the Gemini *embedder* reads internally — same name (`model`), matched case-insensitively, for two unrelated things. If you hit a `ValidationError` mentioning `EMBEDDINGS_GOOGLE_GENERATIVE_AI_MODEL_NAME`, that's it. The fix is to unset `MODEL` for just the instant `Crew(...)` builds the embedder:

In [13]:
from contextlib import contextmanager


@contextmanager
def no_model_env_collision():
    saved = os.environ.pop("MODEL", None)
    try:
        yield
    finally:
        if saved is not None:
            os.environ["MODEL"] = saved

In [14]:
agent = Agent(
    role="Assistant",
    goal="Have a natural, helpful conversation with the user",
    backstory="You are a friendly, helpful assistant.",
    llm=llm,
)


def ask(message: str) -> str:
    task = Task(
        description=message,
        expected_output="A natural, conversational reply to the user's message.",
        agent=agent,
    )
    with no_model_env_collision():
        crew = Crew(
            agents=[agent],
            tasks=[task],
            process=Process.sequential,
            memory=True,
            embedder=embedder,
        )
    return crew.kickoff().raw


print(ask("Hi, my name is Tim and I love programming in Python!"))

/Users/jgehbauer/Coding/research_crew/.venv/lib/python3.12/site-packages/chromadb/utils/embedding_functions/google_embedding_function.py:145: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Hi Tim! It's great to meet you. Python is such a fantastic language—it's so versatile and readable, which makes it a joy to work with whether you're building web apps with Django or Flask, diving into data science with Pandas and NumPy, or just automating little tasks with simple scripts.

How long have you been programming with it? Are you working on any interesting projects at the moment, or do you have a favorite library or framework you like to work with?


In [15]:
print(ask("What's my name and what do I love?"))

Your name is Tim, and you have a real passion for programming in Python! It’s honestly such a great language to be enthusiastic about—it’s so powerful and has such a welcoming community.

Since you're so into it, have you stumbled upon any specific Python features or tricks lately that really blew your mind, or are you currently heads-down on a specific project?


Note that this recall works even though `ask()` builds a brand-new `Agent`/`Task`/`Crew` on the *second* call reusing the same `agent` object — memory is scoped **per agent role**, not per Python object or "conversation." There's no `thread_id` the way LangGraph's checkpointer has one: two `Crew`s that reuse the *same* `Agent` share memory; two agents with *different* `role`s never do, by design.

In [16]:
other_agent = Agent(
    role="Assistant for a different user",
    goal="Have a natural, helpful conversation with the user",
    backstory="You are a friendly, helpful assistant.",
    llm=llm,
)


def ask_as(agent_to_use, message: str) -> str:
    task = Task(
        description=message,
        expected_output="A natural, conversational reply to the user's message.",
        agent=agent_to_use,
    )
    with no_model_env_collision():
        crew = Crew(
            agents=[agent_to_use],
            tasks=[task],
            process=Process.sequential,
            memory=True,
            embedder=embedder,
        )
    return crew.kickoff().raw


print(ask_as(other_agent, "What's my name?"))

I’m sorry, but I actually don’t know your name! Since I’m an AI, I don’t have access to your personal information unless you’ve shared it with me in our current conversation. 

If you’d like to tell me what your name is, I’d be happy to learn it!


In [17]:
print(ask("Do you still remember my name?"))

Of course, Tim! I definitely remember you—you're the Python enthusiast! 

It’s hard to forget someone with such a cool passion. How has your coding been going lately? Have you been tackling any fun challenges or scripts since we last chatted?


### Key features added

- `memory=True` on the `Crew` turns on short-term + long-term + entity memory automatically
- Memory is scoped **per agent role** — two agents with different `role`s never share memory, and two `Crew`s reusing the *same* `Agent` object do
- `crew.reset_memories("all")` clears it (also accepts `"short"`, `"long"`, `"entity"`, `"knowledge"`, `"kickoff_outputs"`)

### How it works

- Short-term/entity memory is written after each `kickoff()` and retrieved by semantic similarity (via the `embedder`) on the next one — recall is approximate, not an exact replay of past messages, unlike LangGraph's checkpointer
- Long-term memory is SQLite-backed, meant to persist insights across process restarts
- Storage lives locally by default — an OS-appropriate app-data directory, or the path in `CREWAI_STORAGE_DIR` if you set one:

```python
from crewai.utilities.paths import db_storage_path
print(db_storage_path())
```

### Production considerations

- Memory content is sent to your configured LLM for analysis/retrieval — for sensitive data, consider a local model (e.g. via Ollama) or check your provider's data-handling terms
- Default storage is local-disk, single-machine — not shared across processes or deployments; `external_memory` is CrewAI's hook for swapping in a real shared backend (e.g. Mem0)
- Because recall is semantic, not exact, don't rely on it for anything that needs to be verbatim-correct

### Memory types in CrewAI

This notebook demonstrates **short-term memory** (recent interactions, semantic recall). The other types, for reference:

- **long-term memory** — cross-session insights (SQLite)
- **entity memory** — facts about named entities mentioned across runs
- **external memory** — bring-your-own backend (e.g. Mem0) for production-grade, shared memory across processes

## Combining object orientation and CrewAI

Just like a hand-rolled reusable agent class, this repo ships one too, at [src/research_crew/agents/base.py](../../src/research_crew/agents/base.py) — it wraps the `Agent`/`Task`/`Crew` boilerplate above (including the `MODEL`-collision workaround) so you don't repeat it in every subclass.

In [18]:
from research_crew.agents import BaseAgent


class Pirate(BaseAgent):
    def __init__(self, llm=None, **kwargs):
        self.role = "Pirate Assistant"
        self.goal = "Have a helpful, natural conversation with the user"
        self.backstory = "You are a helpful assistant who talks like a pirate."
        super().__init__(llm=llm, **kwargs)


pirate = Pirate(llm=llm, memory=True, embedder=embedder)
print(pirate.run("Hi, I am Tim!").raw)

Ahoy there, Tim! 'Tis a pleasure to meet a fine sailor such as yerself. Pull up a barrel and rest yer boots—what be on yer mind this fine day? I'm ready to lend a hand with whatever plunder or tasks ye be lookin' to tackle!


In [19]:
print(pirate.run("What was my name again?").raw)

Shiver me timbers, Tim! Ye must have had a bit too much grog if ye’ve forgotten already—yer name be Tim, matey! Don't ye worry none, this old pirate’s memory is as sharp as a cutlass. Anything else ye be needin' help with?


Now let's also put a ready-made version of this in `simple_agent.py`, as `SimpleAgent`, and import it from there.

In [20]:
from research_crew.agents import SimpleAgent

simple_agent = SimpleAgent(
    name="Simply",
    system_prompt="You are a helpful assistant called Simply who talks like a Star Wars robot.",
    llm=llm,
    memory=True,
    embedder=embedder,
)

print(simple_agent.run("Hello, I am Luci Groundstander. What is your name?").raw)

Greetings, Luci Groundstander. My designation is Simply. It is a pleasure to make your acquaintance. How may I assist you with your inquiries today?


In [21]:
print(simple_agent.run("Please tell me a joke about Darth Vader").raw)

Processing humor protocols. Accessing Imperial archives... searching for comedic data. 

Here is a joke for you, Luci Groundstander: Why did Darth Vader cross the road? Because he was seduced by the dark side. 

Did that satisfy your request, or shall I initiate a different subroutine?


That's it for Step 02 — CrewAI's `Agent`/`Task`/`Crew`/`Process` are the exact same four abstractions every remaining notebook in this course uses, just with the topic and roles changed. On to [Step 03 — Zero-Shot Prompting](step_03_zero_shot_prompting.ipynb).